# Volume 1, Chapter 14: Characters, Euler products and the sieve

Exercises first, checks at the end. The computations confirm finite identities from the chapter: character tables, Legendre's formula, the floor sum for $\mu$, and the order of $5$ modulo a power of $2$. Partial sums and products are evidence about convergence, not proofs.


In [ ]:
import math

def primes_upto(n):
    sieve = [True] * (n + 1)
    sieve[0:2] = [False, False]
    for i in range(2, int(n**0.5) + 1):
        if sieve[i]:
            step = i * i
            sieve[step:n + 1:i] = [False] * (((n - step) // i) + 1)
    return [i for i, flag in enumerate(sieve) if flag]

def mobius_upto(n):
    mu = [1] * (n + 1)
    prime = [True] * (n + 1)
    prime[0] = prime[1] = False
    for p in range(2, n + 1):
        if not prime[p]:
            continue
        for k in range(p, n + 1, p):
            prime[k] = False if k != p else True
            mu[k] *= -1
        square = p * p
        for k in range(square, n + 1, square):
            mu[k] = 0
    mu[0] = 0
    return mu

def phi_sieve(x, z):
    primes = [p for p in primes_upto(z) if p <= z]
    divs = [1]
    for p in primes:
        divs += [d * p for d in divs]
    mu = {1: 1}
    for d in divs:
        m = 0
        n = d
        sign = 1
        for p in primes:
            if n % p == 0:
                n //= p
                m += 1
                sign = -sign
                if n % p == 0:
                    sign = 0
                    break
        mu[d] = sign
    return sum(mu[d] * (x // d) for d in divs)


## Exercises

**P14.1 (★).** Build the four Dirichlet characters modulo $5$ from the primitive root $2$, and check that distinct rows are orthogonal.

**P14.2 (★).** Check $\Phi(30,5)=8$ and $\Phi(100,10)=22$, and recover $\pi(100)=25$.

**P14.3 (★★).** Check $\sum_{n\le x}\mu(n)\lfloor x/n\rfloor=1$ for every $x\le 500$.

**P14.4 (★★).** Check $v_2(5^{2^m}-1)=m+2$ for $0\le m\le 8$, and list the units modulo $32$ as $\pm 5^b$.


In [ ]:
# Your code here


## Checks

In [ ]:
# P14.1 Character table modulo 5.
# chi_j(2^k) = exp(2*pi*i*j*k/4)
units = [1, 2, 4, 3]  # 2^0, 2^1, 2^2, 2^3
rows = []
for j in range(4):
    row = []
    for k, n in enumerate(units):
        row.append(complex(round(math.cos(math.pi * j * k / 2), 10),
                           round(math.sin(math.pi * j * k / 2), 10)))
    rows.append(row)

def dot(a, b):
    return sum(x * y.conjugate() for x, y in zip(a, b))

for i in range(4):
    for j in range(4):
        value = dot(rows[i], rows[j])
        expect = 4 if i == j else 0
        assert abs(value - expect) < 1e-8, (i, j, value)
# Legendre row is the real non-principal character: 1, -1, -1, 1 on 1,2,3,4.
# Our columns are ordered 1,2,4,3, so the real row reads 1,-1,1,-1.
legendre_on_powers = [1, -1, 1, -1]
assert all(abs(rows[2][k] - legendre_on_powers[k]) < 1e-8 for k in range(4))

# P14.2 Sieve counts.
assert phi_sieve(30, 5) == 8
assert phi_sieve(100, 10) == 22
assert phi_sieve(50, 5) == 14
primes = primes_upto(100)
assert sum(1 for p in primes if p <= 100) == 25
assert 22 - 1 + sum(1 for p in primes if p <= 10) == 25

# P14.3 Möbius floor identity.
mu = mobius_upto(500)
for x in range(1, 501):
    total = sum(mu[n] * (x // n) for n in range(1, x + 1))
    assert total == 1, (x, total)

# P14.4 Valuation of powers of 5, and units modulo 32.
def v2(n):
    c = 0
    while n % 2 == 0:
        n //= 2
        c += 1
    return c

for m in range(0, 9):
    assert v2(pow(5, 2**m) - 1) == m + 2

mod = 32
powers = [pow(5, b, mod) for b in range(8)]  # order 2^{5-2}=8
units = []
for a in (0, 1):
    for b, p in enumerate(powers):
        units.append(((-1)**a * p) % mod)
assert sorted(units) == [n for n in range(mod) if n % 2 == 1]
squares = {pow(u, 2, mod) for u in units}
assert squares == {n for n in units if n % 8 == 1}

# Partial Euler product for zeta(2), compared with the partial sum. Not a proof.
partial = sum(n**(-2) for n in range(1, 31))
prod = 1.0
for p in primes_upto(30):
    prod *= 1 / (1 - p**(-2))
basel = math.pi**2 / 6
assert partial < prod < basel

print("Chapter 14 checks passed.")
print(f"partial sum to 30: {partial:.5f}")
print(f"Euler product to 30: {prod:.5f}")
print(f"pi^2/6: {basel:.5f}")
